<a href="https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/NB5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression, evaluated via precision@K (K=20, 50) — same metric as the
Week-4 baseline.** This fits the lane because the underlying decision is still "which pages
first," a ranking problem, not a plain classification problem — per the skill's method table,
ranking problems need a classifier's *probability* scored at precision@K, not just a
yes/no label. Logistic Regression is the right starting point (readable, coefficients are
interpretable) before reaching for Random Forest — simplicity earns trust first, complexity
only if the comparison shows it's worth it.

**Fixing a real issue from earlier weeks:** the Week-4 baseline's "underperforming_ctr" label
was same-window (March CTR vs March benchmark) — a legitimate rule, but not a fair supervised
target, since it can't tell you anything a human reviewer couldn't already see by looking at
March data directly. This notebook instead predicts a genuinely forward-looking outcome:
whether a page's CTR *worsens from March to April* — trained only on March features, labeled
by what actually happened in April. That's a real prediction task, not a restatement of the
present.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import pandas as pd, numpy as np, duckdb

rel = "hf://datasets/FlyRank/internship-warehouse"

df_content = pd.read_parquet(f"{rel}/dim_content.parquet", storage_options={"token": hf_token})
df_march = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": hf_token}
)
df_april = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet",
    storage_options={"token": hf_token}
)

print("March:", df_march.shape, "| April:", df_april.shape)

con = duckdb.connect()
con.register("march", df_march)
con.register("april", df_april)


March: (9841378, 31) | April: (10424730, 31)


In [2]:
def month_agg(table):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM {table}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()

march_agg = month_agg("march")
april_agg = month_agg("april")

march_agg["ctr"] = march_agg["clicks"] / march_agg["impressions"].replace(0, np.nan)
april_agg["ctr"] = april_agg["clicks"] / april_agg["impressions"].replace(0, np.nan)

# Only pages visible in BOTH months, with meaningful March volume (same threshold as Week-4 baseline)
panel = march_agg.merge(april_agg, on=["client_hash_id", "content_hash_id"],
                         suffixes=("_march", "_april"))
panel = panel[(panel["impressions_march"] >= 100) & (panel["impressions_april"] > 0)].copy()

# Forward-looking label: did CTR get WORSE from March to April?
panel["is_declining_label"] = (panel["ctr_april"] < panel["ctr_march"]).astype(int)

panel = panel.merge(
    df_content[["client_hash_id", "content_hash_id", "word_count"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)

print("\nPanel shape (pages visible in both months, meaningful March volume):", panel.shape)
print("Declining rate (April worse than March):", round(panel["is_declining_label"].mean(), 3))
print("Distinct clients in panel:", panel["client_hash_id"].nunique())


Panel shape (pages visible in both months, meaningful March volume): (100893, 12)
Declining rate (April worse than March): 0.442
Distinct clients in panel: 43


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not random.** With 43 distinct clients, a random row-level split would
let the same client appear in both train and test — the model could pick up client-specific
quirks (a particular client's typical position range, content style, etc.) rather than a
generalizable pattern, and the test score would be optimistic in exactly the way the
Week-4/NB2 held-out experiment already demonstrated (0.550 in-sample → 0.400 held-out, same
lesson). This split holds out entire clients, so the test score reflects performance on
clients the model has never seen — the honest number.

No time-based split is needed on top of this, since the forward-looking element is already
built into the label itself (March features → April outcome) rather than needing a
train/test cut across time.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["impressions_march", "avg_position_march", "ctr_march", "word_count"]
X = panel[feature_cols].fillna(0)
y = panel["is_declining_label"].values
groups = panel["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", panel.iloc[train_idx]["client_hash_id"].nunique(),
      "| Test clients:", panel.iloc[test_idx]["client_hash_id"].nunique())
print("Train declining rate:", round(y[train_idx].mean(), 3),
      "| Test declining rate:", round(y[test_idx].mean(), 3))


Train rows: 80891 | Test rows: 20002
Train clients: 30 | Test clients: 13
Train declining rate: 0.443 | Test declining rate: 0.438


Grouped split confirmed: 30 clients in train, 13 in test, with no overlap. Declining rates
are close between train (0.443) and test (0.438), so the split isn't accidentally
concentrating easy or hard cases on one side — a reasonable, representative split.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
from sklearn.linear_model import LogisticRegression

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# --- Model: Logistic Regression ---
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
model_scores_test = model.predict_proba(X_test)[:, 1]

# --- Baseline: same "underperforming CTR" rule as Week-4, benchmark fit on TRAIN only ---
train_panel = panel.iloc[train_idx].copy()
test_panel = panel.iloc[test_idx].copy()

train_panel["position_bucket"] = pd.cut(train_panel["avg_position_march"],
                                         bins=[0, 3, 10, 20, 999], labels=["1-3", "4-10", "11-20", "20+"])
test_panel["position_bucket"] = pd.cut(test_panel["avg_position_march"],
                                        bins=[0, 3, 10, 20, 999], labels=["1-3", "4-10", "11-20", "20+"])

benchmark_pool = train_panel[train_panel["impressions_march"] >= 100]
ctr_benchmark_by_bucket = benchmark_pool.groupby("position_bucket", observed=True)["ctr_march"].mean()

test_panel["ctr_benchmark"] = test_panel["position_bucket"].map(ctr_benchmark_by_bucket).astype(float)
baseline_scores_test = (test_panel["ctr_march"] < test_panel["ctr_benchmark"]).astype(int) * test_panel["impressions_march"]

# --- Comparison table ---
results = []
for k in (20, 50):
    results.append({
        "method": "Week-4 baseline rule",
        "k": k,
        "precision_at_k": round(precision_at_k(baseline_scores_test.values, y_test, k), 3)
    })
    results.append({
        "method": "Logistic Regression",
        "k": k,
        "precision_at_k": round(precision_at_k(model_scores_test, y_test, k), 3)
    })

results_df = pd.DataFrame(results)
print("Base rate (test):", round(y_test.mean(), 3))
results_df


Base rate (test): 0.438


,method,k,precision_at_k
0,Week-4 baseline rule,20,0.50
1,Logistic Regression,20,0.55
2,Week-4 baseline rule,50,0.54
3,Logistic Regression,50,0.70


Logistic Regression beats the Week-4 baseline rule at both K, on a genuinely held-out set of
clients the model never saw in training:
- Precision@20: baseline 0.50 → LR 0.55 (base rate 0.438)
- Precision@50: baseline 0.54 → LR 0.70 (base rate 0.438)

The gap widens at K=50 rather than narrowing — the model isn't just marginally better at
the very top of the queue, it holds up meaningfully further down the list, which matters
more in practice since a reviewer working through a real queue goes well past the top 20.
Both methods clear the 0.438 base rate comfortably, so neither result is an artifact of
class imbalance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
from sklearn.inspection import permutation_importance

# --- What does the model lean on? ---
coefs = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print("Logistic Regression coefficients (sign + magnitude):")
print(coefs)

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
perm_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
print("\nPermutation importance (AUC drop when shuffled):")
print(perm_df)

# --- Where is the model most wrong? ---
test_panel = test_panel.reset_index(drop=True)
test_panel["y_true"] = y_test
test_panel["model_score"] = model_scores_test
test_panel["predicted"] = (test_panel["model_score"] >= 0.5).astype(int)

false_positives = test_panel[(test_panel["predicted"] == 1) & (test_panel["y_true"] == 0)]
false_negatives = test_panel[(test_panel["predicted"] == 0) & (test_panel["y_true"] == 1)]

print(f"\nFalse positives: {len(false_positives)} | False negatives: {len(false_negatives)}")

print("\n3 concrete false positives (model said declining, actually improved/flat):")
print(false_positives.sort_values("model_score", ascending=False)
      [["content_hash_id", "impressions_march", "avg_position_march", "ctr_march", "ctr_april", "model_score"]]
      .head(3))

print("\n3 concrete false negatives (model said fine, actually declined):")
print(false_negatives.sort_values("model_score", ascending=True)
      [["content_hash_id", "impressions_march", "avg_position_march", "ctr_march", "ctr_april", "model_score"]]
      .head(3))


Logistic Regression coefficients (sign + magnitude):
              feature  coefficient
2           ctr_march    46.615980
1  avg_position_march    -0.036175
3          word_count     0.000076
0   impressions_march     0.000049

Permutation importance (AUC drop when shuffled):
              feature  importance_mean  importance_std
2           ctr_march         0.113099        0.001693
1  avg_position_march         0.062958        0.003334
0   impressions_march         0.017211        0.000864
3          word_count         0.003936        0.000545

False positives: 2069 | False negatives: 3920

3 concrete false positives (model said declining, actually improved/flat):
                content_hash_id  impressions_march  avg_position_march  \
11593  content_eadb33b5df496f4a           617124.0            2.383011   
1526   content_8d7d99f109e19aa2           203497.0            2.563756   
1553   content_4ffe18112a5642e3           186983.0            2.331060   

       ctr_march  ctr_april

**What the model leans on:** `ctr_march` dominates by a wide margin — both in coefficient
magnitude (46.6, more than 1000x the next feature) and permutation importance (0.113 AUC
drop when shuffled, vs. 0.063 for the second-place feature). The two rankings agree with
each other (`ctr_march` > `avg_position_march` > `impressions_march` > `word_count` in both),
which is the sanity check the skill asks for — a dominant top feature that's *consistent*
across two different importance methods is a good sign, not the "suspiciously perfect"
pattern that signals leakage. `impressions_march` and `word_count` contribute almost nothing
in either ranking (coefficients near zero, permutation importance near zero).

The sign makes sense as a genuine pattern, not an artifact: `ctr_march`'s positive
coefficient means *higher* March CTR predicts a *higher* chance of declining by April. This
reads as a ceiling/mean-reversion effect — a page already converting well has more room to
fall than one that's already near zero, and per Signal 2's confirmed CTR-vs-position pattern
from Week-4, pages ranking well (which tend to have the highest CTR) are also the ones with
the most to lose if performance regresses even slightly.

**Where the model is wrong — false positives (2,069 cases):** the three largest examples are
all high-volume, top-position pages (positions 2.3–2.6, up to 617k impressions) where the
model assigned near-certain decline (score ~1.0) purely because their March CTR was already
strong. In reality, April CTR held essentially flat or ticked up slightly (e.g., 0.009185 →
0.009300). The model is overconfident on the ceiling-effect logic — it treats "already high"
as "must regress," when small positive fluctuations at that scale are common and don't
represent a real decline worth flagging.

**Where the model is wrong — false negatives (3,920 cases):** the three largest examples are
low-volume, very poorly-ranked pages (avg position 79–91, well outside page 1) with already
weak March CTR, that the model scored as low-risk (0.04–0.05). These pages actually
collapsed to zero clicks in April. The model's logic ("already low, not much room to fall")
breaks down here — it doesn't account for a page losing visibility entirely rather than
just converting slightly worse. This is a real blind spot: the model treats low CTR as a
floor, but for poorly-ranked pages, the real risk is falling off the results page altogether,
which this feature set doesn't capture directly.

False negatives (3,920) outnumber false positives (2,069) roughly 2:1, meaning the model is
more prone to missing real decliners than to over-flagging healthy pages — worth noting for
a reviewer relying on this queue, since it means some real problems will currently slip
through undetected.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.